In [1]:
import datetime
import pandas as pd

In [2]:
full_charging_df = pd.read_feather('../data/full_charging_df.feather')
full_charging_df.head()

,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,...,connectionWeekdayName,connectionYear,isWeekend,connectionHour,disconnectHour,doneChargingHour,loading_duration,connected_duration,ratio_loading_to_connected,invalidDoneDisconnectTimespan
0,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,Thursday,2020,False,5,11,9,262.683333,362.350000,0.724944,False
1,5e23b149f9af8b5fe4b973d0,2020-01-02 05:36:50-08:00,2020-01-02 14:38:21-08:00,2020-01-02 12:18:05-08:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,1,AG-1F01,1-1-193-825,America/Los_Angeles,...,Thursday,2020,False,5,14,12,401.250000,541.516667,0.740974,False
2,5e23b149f9af8b5fe4b973d1,2020-01-02 05:56:35-08:00,2020-01-02 16:39:22-08:00,2020-01-02 08:35:06-08:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,1,AG-1F03,1-1-193-829,America/Los_Angeles,...,Thursday,2020,False,5,16,8,158.516667,642.783333,0.246610,False
3,5e23b149f9af8b5fe4b973d2,2020-01-02 05:59:58-08:00,2020-01-02 08:38:39-08:00,2020-01-02 07:18:45-08:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,1,AG-1F04,1-1-193-820,America/Los_Angeles,...,Thursday,2020,False,5,8,7,78.783333,158.683333,0.496481,False
4,5e23b149f9af8b5fe4b973d3,2020-01-02 06:00:01-08:00,2020-01-02 14:08:40-08:00,2020-01-02 10:17:30-08:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,1,AG-1F06,1-1-193-819,America/Los_Angeles,...,Thursday,2020,False,6,14,10,257.483333,488.650000,0.526928,False


In [3]:
# Convert to UTC because i somehow get errors else
full_charging_df['connectionTime_utc'] = full_charging_df['connectionTime'].dt.tz_convert('UTC')
full_charging_df['disconnectTime_utc'] = full_charging_df['disconnectTime'].dt.tz_convert('UTC')

full_charging_df['all_hours_utc'] = full_charging_df.apply(
    lambda row: pd.date_range(
        start=row['connectionTime_utc'].floor('h'),
        end=row['disconnectTime_utc'].ceil('h'),
        freq='h',
        tz='UTC'
    ),
    axis=1
)

hourly_df = full_charging_df.explode('all_hours_utc').reset_index(drop=True)
hourly_df['hour_start_utc'] = hourly_df['all_hours_utc']
hourly_df['hour_end_utc'] = hourly_df['hour_start_utc'] + pd.Timedelta(hours=1)
hourly_df['burbank_hour'] = hourly_df['hour_start_utc'].dt.tz_convert('America/Los_Angeles')

In [4]:
hourly_df.head()

,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,...,loading_duration,connected_duration,ratio_loading_to_connected,invalidDoneDisconnectTimespan,connectionTime_utc,disconnectTime_utc,all_hours_utc,hour_start_utc,hour_end_utc,burbank_hour
0,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,262.683333,362.35,0.724944,False,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 13:00:00+00:00,2020-01-02 13:00:00+00:00,2020-01-02 14:00:00+00:00,2020-01-02 05:00:00-08:00
1,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,262.683333,362.35,0.724944,False,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 14:00:00+00:00,2020-01-02 14:00:00+00:00,2020-01-02 15:00:00+00:00,2020-01-02 06:00:00-08:00
2,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,262.683333,362.35,0.724944,False,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 15:00:00+00:00,2020-01-02 15:00:00+00:00,2020-01-02 16:00:00+00:00,2020-01-02 07:00:00-08:00
3,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,262.683333,362.35,0.724944,False,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 16:00:00+00:00,2020-01-02 16:00:00+00:00,2020-01-02 17:00:00+00:00,2020-01-02 08:00:00-08:00
4,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,262.683333,362.35,0.724944,False,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:00:00+00:00,2020-01-02 17:00:00+00:00,2020-01-02 18:00:00+00:00,2020-01-02 09:00:00-08:00


## Counting the number of connections per hour

In [5]:
#This dose not include the hours where no one is charging
kpi_df = hourly_df.groupby('burbank_hour').agg(
    session_count=('sessionID', 'nunique'),
).reset_index()

kpi_df

,burbank_hour,session_count
0,2018-04-25 04:00:00-07:00,1
1,2018-04-25 05:00:00-07:00,1
2,2018-04-25 06:00:00-07:00,3
3,2018-04-25 07:00:00-07:00,8
4,2018-04-25 08:00:00-07:00,22
...,...,...
23672,2021-09-14 04:00:00-07:00,1
23673,2021-09-14 05:00:00-07:00,1
23674,2021-09-14 06:00:00-07:00,1
23675,2021-09-14 07:00:00-07:00,1


## Mean delivered kWh per hour

In [6]:
hourly_df['overlap_start_utc'] = hourly_df[['connectionTime_utc', 'hour_start_utc']].max(axis=1)
hourly_df['overlap_end_utc']   = hourly_df[['disconnectTime_utc', 'hour_end_utc']].min(axis=1)
hourly_df['doneChargingTime_utc'] = hourly_df['doneChargingTime'].dt.tz_convert('UTC')
hourly_df['minutes_in_hour'] = (
        (hourly_df['overlap_end_utc'] - hourly_df['overlap_start_utc'])
        .dt.total_seconds() / 60.0
)

# If the loading_duration or minutes_in_hour is zero, the session is not counted. In the function using this data missing data is handled
hourly_df = hourly_df[hourly_df['minutes_in_hour'] > 0]
hourly_df = hourly_df[hourly_df['loading_duration'] > 0]

hourly_df['fraction_of_session'] = hourly_df['minutes_in_hour'] / hourly_df['loading_duration']
hourly_df['kWh_in_this_hour']    = hourly_df['kWhDelivered'] * hourly_df['fraction_of_session']
hourly_df['local_hour'] = hourly_df['hour_start_utc'].dt.tz_convert('America/Los_Angeles')


In [7]:
kpi_df = hourly_df.groupby('local_hour').agg(
    total_kWh=('kWh_in_this_hour', 'sum'),
    session_count=('sessionID', 'nunique'),
)

kpi_df = kpi_df.reset_index()

## Loading Utilization per Hour

In [8]:
def get_is_loading(time: pd.Timestamp, done_charging_time: pd.Timestamp) -> bool:
    if time.hour == done_charging_time.hour:
        return done_charging_time.minute > 29
    return (time + pd.Timedelta(hours=1)) < done_charging_time

In [9]:
hourly_df["loading_utilization"] = hourly_df.apply(
    lambda row: get_is_loading(row["hour_start_utc"], row["doneChargingTime_utc"]),
    axis=1
)

In [10]:
kpi_df = hourly_df.groupby('local_hour').agg(
    total_kWh=('kWh_in_this_hour', 'sum'),
    session_count=('sessionID', 'nunique'),
    loading_utilization=('loading_utilization', 'mean')
)

kpi_df = kpi_df.reset_index()

In [11]:
kpi_df

,local_hour,total_kWh,session_count,loading_utilization
0,2018-04-25 04:00:00-07:00,3.094930,1,1.000000
1,2018-04-25 05:00:00-07:00,3.575657,1,1.000000
2,2018-04-25 06:00:00-07:00,3.161296,3,0.666667
3,2018-04-25 07:00:00-07:00,12.883500,7,1.000000
4,2018-04-25 08:00:00-07:00,41.767757,22,0.954545
...,...,...,...,...
23097,2021-09-14 03:00:00-07:00,5.963001,1,1.000000
23098,2021-09-14 04:00:00-07:00,5.963001,1,1.000000
23099,2021-09-14 05:00:00-07:00,5.963001,1,1.000000
23100,2021-09-14 06:00:00-07:00,5.963001,1,1.000000


In [12]:
def get_kpis(time: datetime.datetime, df: pd.DataFrame):
    """
    Given a Burbank-localized datetime, return the row of KPIs for that hour.
    Assumes df['local_hour'] is also in America/Los_Angeles time.
    """
    row = df.loc[df["local_hour"] == time]
    if row.empty:
        return None
    # Return a single dict with the KPI values
    return {
        "session_count": row["session_count"].iloc[0],
        "total_kWh": row["total_kWh"].iloc[0],
        "loading_utilization": row["loading_utilization"].iloc[0],
    }

In [13]:
specific_time = pd.Timestamp('2019-08-18 22:00:00-08:00')
get_kpis(specific_time, kpi_df)

{'session_count': np.int64(1),
 'total_kWh': np.float64(3.203105590062112),
 'loading_utilization': np.float64(0.0)}

In [14]:
kpi_df.to_feather('../data/kpi_df.feather')